In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import numpy as np
import scipy
import copy

from scipy.sparse import coo_matrix, block_diag, identity, hstack, csr_matrix, csc_matrix, vstack
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import time 

from pyiga import assemble, bspline, vform, geometry, vis, solvers, utils, topology, ieti, algebra, operators, adaptive
from pyiga import algebra_cy, ieti_cy, bspline_cy

from scipy.sparse.linalg import aslinearoperator as LinOp

np.set_printoptions(linewidth=100000)
np.set_printoptions(precision=5)

from sksparse.cholmod import cholesky

In [2]:
def Inductor(deg,N, airgap=0.025):
    kvs=42*[2*(bspline.make_knots(deg,0.0,1.0,N),)]
    
    geos=[      
        geometry.unit_square().scale((0.5)).translate((-0.5,-0.5)),
        geometry.unit_square().scale((0.25,0.5)).translate((0,-0.5)),
        geometry.unit_square().scale((0.25,0.5)).translate((0.25,-0.5)),
        geometry.unit_square().scale((0.5,0.5)).translate((0.5,-0.5)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.,-0.5)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.25,-0.5)),
        geometry.unit_square().scale(0.5).translate((1.5,-0.5)),
        
        geometry.unit_square().scale((0.5,0.25)).translate((-0.5,0)),
        geometry.unit_square().scale(0.25),
        geometry.unit_square().scale(0.25).translate((0.25,0)),
        geometry.unit_square().scale((0.5,0.25)).translate((0.5,0)),
        geometry.unit_square().scale((0.25,0.25)).translate((1.,0)),
        geometry.unit_square().scale((0.25,0.25)).translate((1.25,0)),
        geometry.unit_square().scale((0.5,0.25)).translate((1.5,0)),
        
        geometry.unit_square().scale((0.5,airgap)).translate((-0.5,0.25)),
        geometry.unit_square().scale((0.25,airgap)).translate((0,0.25)),
        geometry.unit_square().scale((0.25,airgap)).translate((0.25,0.25)),
        geometry.unit_square().scale((0.5,airgap)).translate((0.5,0.25)),
        geometry.unit_square().scale((0.25,airgap)).translate((1.,0.25)),
        geometry.unit_square().scale((0.25,airgap)).translate((1.25,0.25)),
        geometry.unit_square().scale((0.5,airgap)).translate((1.5,0.25)),
        
        geometry.unit_square().scale((0.5,0.5)).translate((-0.5,0.25+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((0,0.25+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((0.25,0.25+airgap)),
        geometry.unit_square().scale((0.5,0.5)).translate((0.5,0.25+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.,0.25+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.25,0.25+airgap)),
        geometry.unit_square().scale((0.5,0.5)).translate((1.5,0.25+airgap)),
        
        geometry.unit_square().scale((0.5,0.25)).translate((-0.5,0.75+airgap)),
        geometry.unit_square().scale((0.25,0.25)).translate((0,0.75+airgap)),
        geometry.unit_square().scale((0.25,0.25)).translate((0.25,0.75+airgap)),
        geometry.unit_square().scale((0.5,0.25)).translate((0.5,0.75+airgap)),
        geometry.unit_square().scale((0.25,0.25)).translate((1.,0.75+airgap)),
        geometry.unit_square().scale((0.25,0.25)).translate((1.25,0.75+airgap)),
        geometry.unit_square().scale((0.5,0.25)).translate((1.5,0.75+airgap)),
        
        geometry.unit_square().scale(0.5).translate((-0.5,1.0+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((0,1.0+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((0.25,1.0+airgap)),
        geometry.unit_square().scale((0.5,0.5)).translate((0.5,1.0+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.,1.0+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.25,1.0+airgap)),
        geometry.unit_square().scale(0.5).translate((1.5,1.0+airgap)),
         ]
    patches=list(zip(kvs,geos))
    M = topology.MultiPatch(patches)
    M.rename_domain(0,'Air')
    M.set_domain_id({'Fe':{8,9,10,11,12,22,24,26,29,30,31,32,33}, 'C1':{23}, 'C2':{25}})
    M.h_refine({i:1 for i in range(14,21)});                                        #split airgap patches further in y-axis to make up for anisotropy 
    M.h_refine({i:1 for i in list(range(14,21))+list(range(42,49))});               #split airgap patches further in y-axis to make up for anisotropy 
    #M.h_refine({i:1 for i in list(range(14,21))+list(range(42,49))+list(range(49,63))}); #split airgap patches further in y-axis to make up for anisotropy
    return M

In [3]:
M = Inductor(2,32)
MB = assemble.MultiBasis(M, subspace='C0')

setting up constraints took 0.17643499374389648 seconds.
Basis setup took 0.0074596405029296875 seconds


In [4]:
dir_bcs = MB.set_fixed_boundary({0:0})
Kh = MB.assemble_volume('inner(grad(u),grad(v))*dx', arity=2)
Fh = MB.assemble_volume('f*v*dx', arity=1, f=1)
LS=assemble.RestrictedLinearSystem(Kh,Fh,dir_bcs)

In [22]:
import sksparse.cholmod
A = LS.A.tocsc()
t = time.time()
solver = sksparse.cholmod.cho_factor(A)
print(A@solver.solve(LS.b)-LS.b)
print(time.time()-t)

[ 9.48677e-20  1.62630e-19  5.42101e-20 ...  2.63700e-17 -3.41391e-16  9.62206e-17]
0.3639054298400879


In [25]:
t = time.time()
solver = operators.make_solver(LS.A, spd=False)
print(A@solver(LS.b)-LS.b)
print(time.time()-t)

[ 5.42101e-20 -5.42101e-20 -1.08420e-19 ... -9.15912e-17 -1.52633e-17 -2.36846e-16]
0.27927470207214355


In [2]:
import numpy as np
from scipy.sparse import csc_matrix
from sksparse.cholmod import cholesky

A = csc_matrix([[4., 1.],
                [1., 3.]])

factor = cholesky(A)

b = np.array([1., 2.])
x = factor(b)

print(x)

[0.09091 0.63636]


In [3]:
import ctypes.util
print(ctypes.util.find_library("blas"))
print(ctypes.util.find_library("lapack"))

/home/styoler/miniforge3/lib/libopenblas.so.0
/home/styoler/miniforge3/lib/libopenblas.so.0


In [4]:
from threadpoolctl import threadpool_info
print(threadpool_info())

[{'user_api': 'blas', 'internal_api': 'openblas', 'num_threads': 12, 'prefix': 'libopenblas', 'filepath': '/home/styoler/miniforge3/lib/libopenblasp-r0.3.28.so', 'version': '0.3.28', 'threading_layer': 'pthreads', 'architecture': 'Zen'}, {'user_api': 'openmp', 'internal_api': 'openmp', 'num_threads': 12, 'prefix': 'libomp', 'filepath': '/home/styoler/miniforge3/lib/libomp.so', 'version': None}, {'user_api': 'blas', 'internal_api': 'mkl', 'num_threads': 6, 'prefix': 'libmkl_rt', 'filepath': '/home/styoler/miniforge3/lib/libmkl_rt.so.3', 'version': '2026.0-Product', 'threading_layer': 'gnu'}]
